# Reading with GDAL for Python

This notebook demonstrates some minimal raster/spatial operations with AVIRIS-NG files using the Python interface to GDAL.

<h2 id="tocheading">Table of Contents</h2>       
<br>
<div id="toc"></div>     

*This next cell calls a script to generate a TOC. It will display above when this notebook is opened in the Jupyter environment. Ignore.*

In [1]:
%%javascript
$.getScript('scripts/tocgen.js')

<IPython.core.display.Javascript object>

## Workflow

### Imports and example file
Import requirements. Minimal packages:

In [3]:
import numpy as np
from osgeo import gdal, osr

#### Unzip and open tarfile
AVIRIS-NG data files are distributed in a zipped tarfile. See the document `<link>` for more details. 

You can unzip the example file with the `tarfile` module like this:
```python
import tarfile

with tarfile.open("data/ang20180814t224053rfl.tar.gz", "r:gz") as tar:
    tar.extractall()
```

In [ ]:
import tarfile

with tarfile.open("data/ang20180814t224053rfl.tar.gz", "r:gz") as tar:
    tar.extractall()

See what's inside:

In [4]:
import glob
glob.glob("data/ang20180814t224053_rfl_v2r2/*")

[]

### Reading an image
Open the example reflectance file:

In [23]:
# img = 'data/ang20220710t004851rfl/ang20220710t004851_rfl_v2aa2_img'
# hdr = 'data/ang20220710t004851rfl/ang20220710t004851_rfl_v2aa2_img.hdr'

img = 'data/ang20190713t030616_rfl_v2v2/ang20190713t030616_rfl_v2v2_img'
hdr = 'data/ang20190713t030616_rfl_v2v2/ang20190713t030616_rfl_v2v2_img.hdr'


ds = gdal.Open(img)

Open the header file:

In [24]:
with open(hdr,'r') as f:
    hdr = [ln.strip() for ln in f.readlines()]

### Raster image shape
GDAL makes accessing the shape of the image pretty easy:

In [25]:
bands = ds.RasterCount # band count
cols = ds.RasterXSize  # col count
rows = ds.RasterYSize  # row count

print("bands:\t"+str(bands)) 
print("cols:\t"+str(cols))
print("rows:\t"+str(rows))

bands:	425
cols:	686
rows:	8674


### Geotransform
The geotransform is a tuple of parameters (6 floats) that defines the transformation from each pixel's x,y position in the image to its projected position in the reference coordinate system (affine transformation). 

**This website gives a clear, thorough explanation of affine transforms and their use in GIS:**         
http://www.quantdec.com/GIS/affine.htm

**More info on geographic transformation and GDAL's raster data model:**       
https://www.gdal.org/gdal_datamodel.html

Get the tuple with `ds.GetGeoTransform()`:

In [26]:
ds.GetGeoTransform()

(662183.355625,
 2.9441746679789764,
 -3.269837232100535,
 7628172.98318,
 -3.269837232100535,
 -2.9441746679789764)

#### Calculate pixel coordinates
The coordinates for each pixel are calculated like:
```
GT = ( 447779.369091,                        <-   0 x minimum (top left)
       4.177675425873858,                    <-   1 x resolution
       2.9252398253903347,                   <-   2 x rotation
       7185907.49943,                        <-   3 y maximum (top left)
       2.9252398253903347,                   <-   4 y rotation
      -4.177675425873858 )                   <-   5 y resolution

Xpixel                                       <-  x/column index of pixel
Yline                                        <-  y/row index of pixel

Xproj  = GT(0) + Xpixel*GT(1) + Yline*GT(2)  <-  x coordinate
Yproj  = GT(3) + Xpixel*GT(4) + Yline*GT(5)  <-  y coordinate
```

Calculate the projected coordinates for the pixel at the bottom right corner `(column == 637, row == 4207)`:

In [27]:
# transformation parameters
GT = ds.GetGeoTransform()
print(GT)

# pixel indices are base 0
Xpixel, Yline = (cols - 1, rows - 1)

# x,y calculate w affine transform equation
Xproj = GT[0] + Xpixel*GT[1] + Yline*GT[2]
Yproj = GT[3] + Xpixel*GT[4] + Yline*GT[5]

print("x (m):\t"+str(Xproj))
print("y (m):\t"+str(Yproj))

(662183.355625, 2.9441746679789764, -3.269837232100535, 7628172.98318, -3.269837232100535, -2.9441746679789764)
x (m):	635840.8169585576
y (m):	7600398.317780631


### Coordinate arrays

To make a CF compliant netCDF we need:
* 1-dimensional arrays of x and y coordinates (2)
* 2-dimensional arrays of lon and lat coordinates (2)

First generate the arrays of x and y coordinates. Unpack the geotransform into its component parts and make 1-d arrays with their origins at the top left corner of the raster like:
```
Coordinates calculated over interval equal to their resolution:
    
    xpos(i):  x_origin_in_meters + i*x_resolution
    ypos(i):  y_origin_in_meters + i*y_resolution

x_coordinate_array = xpos(sequence 0 to number_of_columns-1)
y_coordinate_array = ypos(sequence 0 to number_of_rows-1)

```

In [28]:
# get the raster geotransform as its component parts
xmin, xres, xrot, ymax, yrot, yres = ds.GetGeoTransform()

# generate coordinate arrays
xarr = np.array([xmin+i*xres for i in range(0,cols)])
yarr = np.array([ymax+i*yres for i in range(0,rows)])

print("x[0]: \t"+str(xarr[0]))
print("y[0]:\t"+str(yarr[0]))

x[0]: 	662183.355625
y[0]:	7628172.98318


### Get 2d arrays of latitudes and longitudes

Use the `osr` package to get the proj4 string from the input raster dataset:

In [29]:
native_srs = osr.SpatialReference()
native_srs.ImportFromWkt(ds.GetProjection())
proj4 = native_srs.ExportToProj4()

proj4

'+proj=utm +zone=5 +datum=WGS84 +units=m +no_defs'

`pyproj` is the Python interface to libproj. Use pyproj to transform the first pixel's `utm x,y -->> lon,lat`:

In [30]:
# DEPRECATED
# DEPRECATED - see next Code Block
from pyproj import Proj, transform

inproj = Proj(proj4)
outproj = Proj(init="epsg:4326")
lon, lat = transform(inproj, outproj, xarr[0], yarr[0])

lon, lat

/Users/bryantutt/miniconda3/envs/GIS_312/lib/python3.12/site-packages/pyproj/crs/crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
/var/folders/8x/h1qrj40n693fk_zzl2_sskt40000gn/T/ipykernel_70638/2059403006.py:7: FutureWarning: This function is deprecated. See: https://pyproj4.github.io/pyproj/stable/gotchas.html#upgrading-to-pyproj-2-from-pyproj-1
  lon, lat = transform(inproj, outproj, xarr[0], yarr[0])


(-148.99372026305792, 68.7177165467529)

In [31]:
# UPDATED version to avoid Warnings from Deprecated code (previous Code Block)
from pyproj import CRS, Transformer

# Define the input and output coordinate reference systems
inproj = CRS(proj4)  # Assuming `proj4` is a valid PROJ string
outproj = CRS.from_epsg(4326)  # WGS84

# Create a transformer for coordinate conversion
transformer = Transformer.from_crs(inproj, outproj, always_xy=True)

# Transform the coordinates
lon, lat = transformer.transform(xarr[0], yarr[0])

lon, lat

(-148.99372026305792, 68.7177165467529)

Permute the x and y arrays with `np.meshgrid`:

In [32]:
xarr2d, yarr2d = np.meshgrid(xarr, yarr)

print("Each array now has this shape:\t"+str(xarr2d.shape))

Each array now has this shape:	(8674, 686)


Flatten both arrays and pass to the `pyproj.transform` function:

In [ ]:
# DEPRECATED
# DEPRECATED - see next Code Block
lonarr, latarr = transform(
    inproj,               # input raster srs
    outproj,              # output raster srs
    xarr2d.flatten(),     # flat 2d array of x coordinates
    yarr2d.flatten())     # flat 2d array of y coordinates

print("lon[0]:\t"+str(lonarr[0]))
print("lat[0]:\t"+str(latarr[0]))

/var/folders/8x/h1qrj40n693fk_zzl2_sskt40000gn/T/ipykernel_70638/3706868637.py:1: FutureWarning: This function is deprecated. See: https://pyproj4.github.io/pyproj/stable/gotchas.html#upgrading-to-pyproj-2-from-pyproj-1
  lonarr, latarr = transform(


lon[0]:	68.72450853393356
lat[0]:	-149.4839367834409


In [34]:
# Updated version to avoid Warnings from Deprecated code (previous Code Block)
from pyproj import CRS, Transformer

# Define input and output CRS
inproj = CRS(proj4)  # Assuming `proj4` is a valid PROJ string
outproj = CRS.from_epsg(4326)  # WGS84

# Create a transformer for coordinate conversion (longitude, latitude order ensured)
transformer = Transformer.from_crs(inproj, outproj, always_xy=True)

# Transform the coordinates (ensuring correct order)
lonarr, latarr = transformer.transform(xarr2d.flatten(), yarr2d.flatten())

# Print the first transformed coordinates
print(f"lon[0]:\t{lonarr[0]}")
print(f"lat[0]:\t{latarr[0]}")

lon[0]:	-148.99372026305792
lat[0]:	68.7177165467529


Return the flat arrays to the shape of the raster:

In [35]:
lonarr2d = lonarr.reshape(xarr2d.shape)
latarr2d = latarr.reshape(yarr2d.shape)

lonarr2d.shape

(8674, 686)